# 01 - Solar Data Exploration

Load solar radiation data using the Solar Intelligence data loader,
explore the xarray Dataset structure, and visualize basic patterns.

In [ ]:
from solar_intelligence.data_loader import generate_synthetic_solar_data, DataLoader
import xarray as xr
import pandas as pd
import numpy as np

## Load Synthetic Solar Data
Generate physics-based synthetic data for New Delhi.

In [ ]:
ds = generate_synthetic_solar_data(lat=28.6139, lon=77.2090, start_year=2020, end_year=2023)
print(ds)

## Explore Dataset Structure

In [ ]:
# Dimensions, coordinates, variables
print(f"Dimensions: {dict(ds.dims)}")
print(f"\nCoordinates: {list(ds.coords)}")
print(f"\nData variables: {list(ds.data_vars)}")
print(f"\nAttributes: {ds.attrs}")

In [ ]:
# Variable-level metadata
for var in ['ALLSKY_SFC_SW_DWN', 'T2M', 'ALLSKY_KT']:
    da = ds[var]
    print(f"{var}: shape={da.shape}, dtype={da.dtype}, attrs={da.attrs}")

## xarray Operations

In [ ]:
# Time slicing
summer_2023 = ds.sel(time=slice('2023-06-01', '2023-08-31'))
print(f"Summer 2023: {len(summer_2023.time)} days")

# Monthly mean using .groupby()
monthly_ghi = ds['ALLSKY_SFC_SW_DWN'].groupby('time.month').mean()
print(f"\nMonthly GHI:\n{monthly_ghi.values.round(2)}")

# Rolling average
rolling_30d = ds['ALLSKY_SFC_SW_DWN'].rolling(time=30, center=True).mean()
print(f"\nRolling 30-day average shape: {rolling_30d.shape}")

In [ ]:
# Resample to monthly
monthly = ds['ALLSKY_SFC_SW_DWN'].resample(time='ME').mean()
print(f"Monthly resampled: {len(monthly)} months")

# Seasonal statistics
seasonal = ds['ALLSKY_SFC_SW_DWN'].groupby('time.season').mean()
print(f"\nSeasonal means: {dict(zip(seasonal.season.values, seasonal.values.round(2)))}")

# Basic statistics
ghi = ds['ALLSKY_SFC_SW_DWN']
print(f"\nGHI stats: mean={float(ghi.mean()):.2f}, std={float(ghi.std()):.2f}, "
      f"min={float(ghi.min()):.2f}, max={float(ghi.max()):.2f}")

## Save as NetCDF

In [ ]:
# Save and reload
ds.to_netcdf('/tmp/solar_test.nc')
ds_loaded = xr.open_dataset('/tmp/solar_test.nc')
print(f"Reloaded: {list(ds_loaded.data_vars)}")